# Feature Engineering Pipeline
**Overview:** Building upon the insights gathered during Exploratory Data Analysis (EDA), this notebook focuses on transforming and aggregating transactional data into a comprehensive feature set ready for machine learning model training. The primary objective is to transition from item-level details to daily time-series data, construct lag features for autocorrelation, and encode complex marketing campaigns to explain revenue fluctuations.

## 1. Setup & Data Loading
In this section, we initialize our analytical environment, import required libraries, and ingest the pre-cleaned datasets for downstream processing.

### 1.1. Import Libraries & Environment Setup

In [1]:
import pandas as pd
import os
import warnings

# Ignore warning messages for cleaner notebook output
warnings.filterwarnings('ignore')

### 1.2. Load Datasets
Loading the cleaned `.parquet` files from the processed dataset directory and web_traffic.

In [2]:
DATA_PATH = '../dataset/03_after_eda/'
print("Loading data...")
df_orders = pd.read_parquet(DATA_PATH + 'order_items_detail.parquet')
df_traffic = pd.read_parquet('../dataset/02_after_clean/web_traffic.parquet')
df_traffic = df_traffic.rename(columns={'date': 'Date'})
df_master = pd.merge(df_orders, df_traffic[['page_views', 'Date']], on='Date', how='left')
print("Load data successfully")

Loading data...
Load data successfully


In [3]:
df_master.head()

,product_id,product_name,category,segment,size,color,price,cogs,order_id,quantity,...,end_date,applicable_category,promo_channel,stackable_flag,min_order_value,Date,Revenue_line,COGS_line,Profit_line,page_views
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.65,9704.842875,245451,5.0,...,NaT,NaN,NaN,None,NaN,2015-01-07,55918.35,48524.214375,7394.135625,33888.0
1,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.65,9704.842875,245898,8.0,...,NaT,NaN,NaN,None,NaN,2015-01-13,87350.00,77638.743000,9711.257000,38118.0
2,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.65,9704.842875,246268,6.0,...,NaT,NaN,NaN,None,NaN,2015-01-13,67107.30,58229.057250,8878.242750,38118.0
3,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.65,9704.842875,246907,2.0,...,NaT,NaN,NaN,None,NaN,2015-01-17,22209.44,19409.685750,2799.754250,50373.0
4,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.65,9704.842875,247911,6.0,...,NaT,NaN,NaN,None,NaN,2015-01-24,66858.96,58229.057250,8629.902750,52366.0


## 2. Feature Engineering Strategy

To surpass the standard baseline, we designed a robust FE pipeline focusing on three core feature groups:
- **Group 1 - Transaction & Product (Item-level):** Normalizing discount types (fixed vs. percentage) and One-Hot Encoding product categories.
- **Group 2 - Historical & Lag (Lag & Rolling Features):** Capturing trend cycles through 1-day, 7-day (weekly cycle), and 365-day (yearly cycle) lags.
- **Group 3 - Events & Promotions (Event/Promo Calendar):** Deconstructing sales events into specific timelines (Spring, Midyear, Fall, Yearend...) enabling the model to distinguish between natural demand and marketing-driven sales spikes.

## 3. Item-Level Processing
Before aggregating data daily, we need to process attributes at the individual product/order level.

### 3.1. Temporal Features
Extract day of the week, month, and identify weekends.

In [4]:
df_master['day_of_week'] = df_master['Date'].dt.dayofweek
df_master['month'] = df_master['Date'].dt.month
df_master['is_weekend'] = df_master['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

### 3.2. Discount Normalization
In the raw dataset, `discount_value` exists in two formats: `fixed` amount and `percentage`. To allow the model to evaluate the discount magnitude consistently, we will convert everything into a standardized percentage format (`discount_pct`).

In [5]:
# Since discount_value has 2 formats (fixed and percentage), we will convert everything to percentage
df_master['discount_value'].value_counts()
df_master['discount_pct'] = 0.0

# Create mask variables to filter the 2 types: fixed and percentage
mask_pct = df_master['promo_type'] == 'percentage'
mask_fixed = df_master['promo_type'] == 'fixed'

# Calculate discount_pct: if percentage, divide by 100; if fixed, divide discount amount by price
df_master.loc[mask_pct, 'discount_pct'] = df_master.loc[mask_pct, 'discount_value'] / 100
df_master.loc[mask_fixed, 'discount_pct'] = df_master.loc[mask_fixed, 'discount_value'] / df_master.loc[mask_fixed, 'price']
df_master['discount_pct'] = df_master['discount_pct'].clip(0, 1)

print("Các giá trị discount_pct sau khi tính toán:")
print(df_master['discount_pct'].value_counts().head(10))

df_master['discount_pct_mean'] = df_master['discount_pct'].mean()
df_master['discount_pct_max'] = df_master['discount_pct'].max()

Các giá trị discount_pct sau khi tính toán:
discount_pct
0.000000    438353
0.200000     71799
0.120000     71591
0.180000     57417
0.100000     45654
0.150000      8905
0.008532       460
0.003840       459
0.008978       404
0.008150       401
Name: count, dtype: int64


### 3.3. One-Hot Encoding for Category & Segment
Transform categorical variables to track the daily sales volume or revenue of specific product groups.

In [6]:
# One-Hot Encoding for Category & Segment
df_encoded = pd.get_dummies(df_master, columns=['category', 'segment'], prefix=['cat', 'seg'])

## 4. Daily Time-Series Aggregation
This is a crucial transformation step: Grouping all transactional data by `Date`. Financial metrics and sales volume by Category/Segment will be summed. Discount attributes will be aggregated using average and maximum functions.

In [7]:
df_daily = df_master.groupby('Date').agg({
    'Revenue_line': 'sum',   
    'day_of_week': 'first',
    'is_weekend': 'first',
}).reset_index()

In [8]:
# Get the list of newly created One-Hot columns
encoded_cols = [col for col in df_encoded.columns if col.startswith(('cat_', 'seg_'))]

# For One-Hot columns, we use 'sum' to know how many items of that type were sold that day
agg_logic = {
    'Revenue_line': 'sum',
    'Profit_line': 'sum',
    'COGS_line': 'sum',
    'day_of_week': 'first',
    'month': 'first',
    'is_weekend': 'first',
    'discount_pct_mean' : 'mean',
    'discount_pct_max' : 'max',
    'page_views': 'first'
}

# Add encoded columns to the aggregation logic
for col in encoded_cols:
    agg_logic[col] = 'sum'

# Apply groupby and aggregation
df_daily = df_encoded.groupby('Date').agg(agg_logic).reset_index()
df_daily.columns = ['_'.join(col) if isinstance(col, tuple) else col for col in df_daily.columns]

## 5. Time-Series Features
After obtaining daily aggregated data, we apply specific techniques for Time-Series forecasting, including historical lag features and refined financial metrics.

### 5.1. Lag Features

In [9]:
# The model will look at the results of 1 day, 7 days, and 365 days prior for forecasting
df_daily = df_daily.sort_values('Date')
df_daily['rev_lag_1'] = df_daily['Revenue_line'].shift(1)
df_daily['rev_lag_7'] = df_daily['Revenue_line'].shift(7)
df_daily['rev_lag_365'] = df_daily['Revenue_line'].shift(365)
df_daily['margin_rate']= df_daily['Profit_line'] / (df_daily['Revenue_line']+1)
df_daily['margin_rate_lag_7']= df_daily['margin_rate'].shift(7)
df_daily['margin_rate_lag_365']= df_daily['margin_rate'].shift(365)
df_daily['profit_lag_7'] = df_daily['Profit_line'].shift(7)
df_daily['profit_lag_365'] = df_daily['Profit_line'].shift(365)
df_daily['page_views_MA_30'] = df_daily['page_views'].rolling(window=30, min_periods=1).mean()
df_daily['page_views_lag_365'] = df_daily['page_views'].shift(365)
df_daily['page_views_MA_30_lag_365'] = df_daily['page_views_MA_30'].shift(365)
# Finally, fill NaN values generated by the shift function
df_daily = df_daily.fillna(0)

### 5.2. Metric Refinement & Renaming

In [10]:

df_daily = df_daily.rename(columns={'Revenue_line': 'Revenue', 'Profit_line': 'Profit', 'COGS_line': 'COGS'})
df_daily = df_daily.rename(columns={'discount_pct_mean': 'avg_discount_pct', 'discount_pct_max': 'max_discount_pct'})

print("Kích thước bảng Daily:", df_daily.shape)
display(df_daily.head())

Kích thước bảng Daily: (3833, 33)


,Date,Revenue,Profit,COGS,day_of_week,month,is_weekend,avg_discount_pct,max_discount_pct,page_views,...,rev_lag_7,rev_lag_365,margin_rate,margin_rate_lag_7,margin_rate_lag_365,profit_lag_7,profit_lag_365,page_views_MA_30,page_views_lag_365,page_views_MA_30_lag_365
0,2012-07-04,5123547.94,1.140557e+06,3.982991e+06,2,7,0,0.055065,0.2,0.0,...,0.0,0.0,0.222611,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2012-07-05,2751773.45,6.011932e+05,2.150580e+06,3,7,0,0.055065,0.2,0.0,...,0.0,0.0,0.218475,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2012-07-06,3054029.42,5.363966e+05,2.517633e+06,4,7,0,0.055065,0.2,0.0,...,0.0,0.0,0.175636,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2012-07-07,2667930.94,5.596843e+05,2.108247e+06,5,7,1,0.055065,0.2,0.0,...,0.0,0.0,0.209782,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2012-07-08,2360851.90,5.522291e+05,1.808623e+06,6,7,1,0.055065,0.2,0.0,...,0.0,0.0,0.233911,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 6. Holiday & Promotion Calendar Mapping

Shopping behavior is heavily influenced by annual events. This section precisely defines the timeframes of major sales campaigns (Spring, Midyear, Yearend...) based on business rules. This prevents the model from interpreting sudden marketing-driven revenue spikes as natural baseline trends.

In [11]:
df_daily['year'] = df_daily['Date'].dt.year
df_daily['month_day'] = df_daily['Date'].dt.strftime('%m-%d')

df_daily['is_Spring_promotion'] = df_daily['month_day'].between('03-18', '04-17').astype(int)
df_daily['is_MidYear_promotion'] = df_daily['month_day'].between('06-23', '07-22').astype(int)
df_daily['is_FallLauch_promotion'] = (df_daily['month_day'].between('08-30', '10-01') | ((df_daily['year'] % 4 == 1) & (df_daily['month_day'] == '10-02'))).astype(int)
df_daily['is_YearEnd_promotion'] = (df_daily['month_day'].between('11-18', '12-31') |  df_daily['month_day'].between('01-01', '01-02')).astype(int) # có nhiều ngày kết thúc khác nhưng fix vào 02/01 hàng năm
df_daily['is_Rural_promotion'] = ((df_daily['year'] % 2 == 1)  & df_daily['month_day'].between('01-30', '03-01')).astype(int) # từ năm 2015 mới cố định ngày kết thúc
df_daily['is_Urban_promotion'] = ((df_daily['year'] % 2 == 1)  & df_daily['month_day'].between('07-30', '09-02')).astype(int)
df_daily['category_promotion'] = -1 # default: if it isn't in any promotion
df_daily['chanel_promotion'] = -1 # default: if it isn't in any promotion
# df_daily['pct_discount_promotion'] = 0.0 # default: if it isn't in any promotion
# df_daily['fix_discount_promotion'] = 0 # default: if it isn't in any promotion

#decode category_promotion: 0 -> All ; 1 -> Outdoor ; 2 -> Steetwear
df_daily.loc[(df_daily['is_Spring_promotion'] | df_daily['is_MidYear_promotion'] | df_daily['is_YearEnd_promotion']), 'category_promotion'] = 0
df_daily.loc[df_daily['is_Rural_promotion'], 'category_promotion'] = 1
df_daily.loc[df_daily['is_Urban_promotion'], 'category_promotion'] = 2

#decode chanel_promotion: 0 -> all_chanel ; 1 -> in_store ; 2 -> online ; 3 -> onl or social ; 4 onl or email or social
df_daily.loc[(df_daily['is_Spring_promotion'] | df_daily['is_FallLauch_promotion']), 'chanel_promotion'] = 4
df_daily.loc[df_daily['is_Rural_promotion'], 'chanel_promotion'] = 1
df_daily.loc[df_daily['is_Urban_promotion'], 'chanel_promotion'] = 2
df_daily.loc[df_daily['is_MidYear_promotion'], 'chanel_promotion'] = 3
df_daily.loc[df_daily['is_YearEnd_promotion'], 'chanel_promotion'] = 0

**Additional Feature:** Extracting the Quarter component from the `Date` column to enhance the model's ability to capture seasonality.

In [12]:
df_daily['quarter'] = df_daily['Date'].dt.quarter

## 7. Feature Selection & Data Export
Dropping irrelevant columns to optimize memory usage, reviewing the final structure of our dataset and exporting the prepared data for the EDA phase.

In [13]:
EXPORT_PATH = '../dataset/04_after_fe/'
os.makedirs(EXPORT_PATH, exist_ok=True)

# Drop operational columns that are no longer needed for high-level segment analysis
df_daily = df_daily.drop(columns=['year', 'month_day'])

# Review dataset schemas and non-null counts
df_daily.info()
# Export the processed dataframes
df_daily.to_parquet(EXPORT_PATH + 'processed_data_daily.parquet', index=False)
print("Data preparation complete.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 42 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Date                      3833 non-null   datetime64[ns]
 1   Revenue                   3833 non-null   float64       
 2   Profit                    3833 non-null   float64       
 3   COGS                      3833 non-null   float64       
 4   day_of_week               3833 non-null   int32         
 5   month                     3833 non-null   int32         
 6   is_weekend                3833 non-null   int64         
 7   avg_discount_pct          3833 non-null   float64       
 8   max_discount_pct          3833 non-null   float64       
 9   page_views                3833 non-null   float64       
 10  cat_Casual                3833 non-null   int64         
 11  cat_GenZ                  3833 non-null   int64         
 12  cat_Outdoor         